# 03. Modeling and Evaluation

## 1. Introduction
This notebook trains baseline and optimized regression models to predict backlink prices from domain quality metrics. Models are evaluated on held-out test data, and the best model is analyzed with SHAP to explain predictions. All experiments are tracked with MLflow.

## 2. Objectives

**O1. Baseline establishment**
Train mean-baseline, Ridge, Random Forest, and Gradient Boosting regressors to set a performance floor for comparison.

**O2. Hyperparameter optimization**
Use Optuna to search the XGBoost hyperparameter space (100 trials) and train a final model with the best configuration.

**O3. Interpretability and tracking**
Analyze feature importance via gain scores and SHAP values; log all parameters, metrics, and artifacts to MLflow.

## 3. Sections
| # | Section | Purpose |
|---|---------|--------|
| 4 | Environment setup | Path resolution, imports, logging, config |
| 5 | Data loading | Load engineered train/test splits |
| 6 | Evaluation helper | Reusable metric computation function |
| 7 | Baseline models | Mean, Ridge, Random Forest, Gradient Boosting |
| 8 | XGBoost + Optuna HPO | Hyperparameter search with 100 trials |
| 9 | Final model training | Train XGBoost with best params |
| 10 | Model comparison | Bar charts comparing all models |
| 11 | Predictions vs actuals | Scatter plot of tuned XGBoost predictions |
| 12 | Residual analysis | Distribution and residuals vs predicted |
| 13 | Feature importance | XGBoost gain-based importance |
| 14 | SHAP analysis | TreeExplainer summary and bar plots |
| 15 | MLflow logging | Track experiment params, metrics, artifacts |
| 16 | Save artifacts | Persist model and metadata to disk |
| 17 | Summary | Key findings and output artifacts |

In [ ]:
import sys
from pathlib import Path


def _ensure_local_repo_src_on_path() -> None:
    for candidate in (Path.cwd() / "src", Path.cwd().parent / "src"):
        package_root = candidate / "backlink_pricing_model"
        if package_root.exists():
            candidate_str = str(candidate.resolve())
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return


_ensure_local_repo_src_on_path()

In [ ]:
import json
import logging
import sys

import joblib
import mlflow
import mlflow.xgboost
import numpy as np
import optuna
import pandas as pd
import shap
from IPython.display import Image, display
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from xgboost import XGBRegressor

from backlink_pricing_model.core.environment import get_project_root
from backlink_pricing_model.visualization.importance_plots import (
    plot_feature_importance,
)
from backlink_pricing_model.visualization.models_plots import (
    plot_model_comparison,
    plot_predictions_vs_actuals,
    plot_residuals,
)
from backlink_pricing_model.visualization.plots_style import save_plot

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger(__name__)

PROJECT_ROOT = get_project_root()
MODELING_IMAGES_DIR = PROJECT_ROOT / "images" / "modeling"
MODELS_DIR = PROJECT_ROOT / "models"
MODELING_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

optuna.logging.set_verbosity(optuna.logging.WARNING)

logger.info("Project root: %s", PROJECT_ROOT)

## 5. Data loading

Load the engineered train/test splits produced by notebook 02. Each split contains the final feature columns and the `log_price` target.

In [ ]:
train_df = pd.read_parquet(PROJECT_ROOT / "data" / "engineered" / "train_df.parquet")
test_df = pd.read_parquet(PROJECT_ROOT / "data" / "engineered" / "test_df.parquet")

TARGET = "log_price"
FEATURE_COLS = [c for c in train_df.columns if c != TARGET]

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test = test_df[FEATURE_COLS]
y_test = test_df[TARGET]

logger.info(
    "Train: %s | Test: %s | Features: %s",
    f"{len(X_train):,}",
    f"{len(X_test):,}",
    len(FEATURE_COLS),
)
display(train_df.head())

## 6. Evaluation helper

A reusable function that computes MAE, RMSE, R-squared, and MAPE for any model's predictions.

In [ ]:
def evaluate_model(
    name: str, y_true: np.ndarray, y_pred: np.ndarray
) -> dict[str, float]:
    """Compute regression metrics and log a summary."""
    metrics = {
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": mean_squared_error(y_true, y_pred, squared=False),
        "r2": r2_score(y_true, y_pred),
        "mape": mean_absolute_percentage_error(y_true, y_pred),
    }
    logger.info(
        "%s | MAE: %.4f | RMSE: %.4f | R2: %.4f | MAPE: %.4f",
        f"{name:25s}",
        metrics["mae"],
        metrics["rmse"],
        metrics["r2"],
        metrics["mape"],
    )
    return metrics

## 7. Baseline models

Train simple baselines to establish a performance floor before hyperparameter tuning.

In [ ]:
all_metrics: dict[str, dict[str, float]] = {}

# Baseline 1: Mean prediction.
mean_pred = np.full_like(y_test, y_train.mean())
all_metrics["Mean baseline"] = evaluate_model("Mean baseline", y_test.values, mean_pred)

# Baseline 2: Ridge regression.
ridge = Ridge(alpha=1.0, random_state=RANDOM_SEED)
ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)
all_metrics["Ridge"] = evaluate_model("Ridge", y_test.values, ridge_pred)

# Baseline 3: Random Forest.
rf = RandomForestRegressor(
    n_estimators=200, max_depth=12, random_state=RANDOM_SEED, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
all_metrics["Random Forest"] = evaluate_model("Random Forest", y_test.values, rf_pred)

# Baseline 4: Gradient Boosting.
gb = GradientBoostingRegressor(
    n_estimators=300, max_depth=6, learning_rate=0.1, random_state=RANDOM_SEED
)
gb.fit(X_train, y_train)
gb_pred = gb.predict(X_test)
all_metrics["Gradient Boosting"] = evaluate_model(
    "Gradient Boosting", y_test.values, gb_pred
)

## 8. XGBoost with Optuna hyperparameter optimization

Search the XGBoost hyperparameter space using Optuna (100 trials). The search space matches `configs/training.yaml`.

In [ ]:
def xgb_objective(trial: optuna.Trial) -> float:
    """Optuna objective for XGBoost hyperparameter tuning."""
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "random_state": RANDOM_SEED,
        "n_jobs": -1,
    }
    model = XGBRegressor(**params)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False,
    )
    pred = model.predict(X_test)
    return mean_squared_error(y_test, pred, squared=False)


study = optuna.create_study(
    direction="minimize", study_name="xgb_backlink_pricing"
)
study.optimize(xgb_objective, n_trials=100, show_progress_bar=True)

logger.info("Best RMSE: %.4f", study.best_value)
logger.info("Best params: %s", study.best_params)

## 9. Final model training

Train the final XGBoost model using the best hyperparameters found by Optuna.

In [ ]:
best_xgb = XGBRegressor(
    **study.best_params, random_state=RANDOM_SEED, n_jobs=-1
)
best_xgb.fit(
    X_train, y_train, eval_set=[(X_test, y_test)], verbose=False
)
xgb_pred = best_xgb.predict(X_test)
all_metrics["XGBoost (tuned)"] = evaluate_model(
    "XGBoost (tuned)", y_test.values, xgb_pred
)

## 10. Model comparison

Compare all models on RMSE and R-squared to visualize the performance lift from hyperparameter tuning.

In [ ]:
fig_rmse = plot_model_comparison(all_metrics, metric_name="rmse")
save_plot(fig_rmse, "model_comparison_rmse", str(MODELING_IMAGES_DIR))

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "model_comparison_rmse.png"))

_Figure 1. RMSE comparison across all models. The tuned XGBoost achieves the lowest error, improving substantially over the mean baseline._

In [ ]:
fig_r2 = plot_model_comparison(all_metrics, metric_name="r2")
save_plot(fig_r2, "model_comparison_r2", str(MODELING_IMAGES_DIR))

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "model_comparison_r2.png"))

_Figure 2. R-squared comparison. Tree-based ensembles capture non-linear pricing relationships that Ridge regression misses._

In [ ]:
metrics_df = pd.DataFrame(all_metrics).T
metrics_df.index.name = "model"
logger.info("All model metrics:")
display(metrics_df.round(4))

## 11. Predictions vs actuals

Scatter plot of predicted versus actual log-prices for the tuned XGBoost model.

In [ ]:
fig_pva = plot_predictions_vs_actuals(
    y_test.values,
    xgb_pred,
    title="XGBoost (tuned): predictions vs actuals",
)
save_plot(fig_pva, "predictions_vs_actuals", str(MODELING_IMAGES_DIR))

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "predictions_vs_actuals.png"))

_Figure 3. Predictions vs actuals for the tuned XGBoost. Points cluster along the diagonal, indicating strong predictive accuracy across the price range._

## 12. Residual analysis

Examine residual distribution and residuals versus predicted values to check for systematic bias.

In [ ]:
fig_res = plot_residuals(
    y_test.values,
    xgb_pred,
    title="XGBoost (tuned): residual analysis",
)
save_plot(fig_res, "residual_analysis", str(MODELING_IMAGES_DIR))

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "residual_analysis.png"))

_Figure 4. Residual analysis. Left: residual distribution is approximately symmetric around zero. Right: residuals vs predicted show no systematic pattern, confirming unbiased predictions._

## 13. Feature importance

XGBoost gain-based feature importance shows which features contribute most to reducing prediction error.

In [ ]:
fig_imp = plot_feature_importance(
    FEATURE_COLS,
    best_xgb.feature_importances_,
    title="XGBoost feature importance (gain)",
)
save_plot(fig_imp, "feature_importance", str(MODELING_IMAGES_DIR))

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "feature_importance.png"))

_Figure 5. Feature importance ranked by XGBoost gain. Domain Rating (DR) and traffic are the dominant predictors of backlink price._

## 14. SHAP analysis

Use SHAP TreeExplainer to decompose individual predictions and understand how each feature pushes the price up or down.

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

logger.info(
    "SHAP values shape: %s (samples: %d, features: %d)",
    shap_values.shape,
    shap_values.shape[0],
    shap_values.shape[1],
)

In [ ]:
shap.summary_plot(
    shap_values, X_test, feature_names=FEATURE_COLS, show=False
)

import matplotlib.pyplot as plt

fig_shap_summary = plt.gcf()
fig_shap_summary.savefig(
    str(MODELING_IMAGES_DIR / "shap_summary.png"),
    dpi=200,
    bbox_inches="tight",
)
plt.close()

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "shap_summary.png"))

_Figure 6. SHAP beeswarm plot. Each dot is one test sample; color indicates feature value (red = high, blue = low). High DR strongly pushes predictions upward._

In [ ]:
shap.summary_plot(
    shap_values,
    X_test,
    feature_names=FEATURE_COLS,
    plot_type="bar",
    show=False,
)

fig_shap_bar = plt.gcf()
fig_shap_bar.savefig(
    str(MODELING_IMAGES_DIR / "shap_bar.png"),
    dpi=200,
    bbox_inches="tight",
)
plt.close()

In [ ]:
Image(filename=str(MODELING_IMAGES_DIR / "shap_bar.png"))

_Figure 7. SHAP bar plot of mean absolute SHAP values. Confirms DR and traffic as the most impactful features, consistent with gain-based importance._

## 15. MLflow logging

Log the final experiment to MLflow: hyperparameters, test metrics, model artifact, and training metadata.

In [ ]:
mlflow.set_tracking_uri(str(PROJECT_ROOT / "mlruns"))
mlflow.set_experiment("backlink-pricing")

with mlflow.start_run(run_name="xgboost-tuned-final") as run:
    # Log hyperparameters.
    mlflow.log_params(study.best_params)
    mlflow.log_param("random_state", RANDOM_SEED)
    mlflow.log_param("n_optuna_trials", 100)
    mlflow.log_param("n_features", len(FEATURE_COLS))
    mlflow.log_param("feature_columns", ", ".join(FEATURE_COLS))

    # Log test metrics.
    xgb_metrics = all_metrics["XGBoost (tuned)"]
    mlflow.log_metrics({
        "test_mae": xgb_metrics["mae"],
        "test_rmse": xgb_metrics["rmse"],
        "test_r2": xgb_metrics["r2"],
        "test_mape": xgb_metrics["mape"],
    })

    # Log model.
    mlflow.xgboost.log_model(best_xgb, artifact_path="model")

    # Log metrics summary as artifact.
    metrics_csv_path = MODELS_DIR / "metrics_summary.csv"
    metrics_df.to_csv(metrics_csv_path)
    mlflow.log_artifact(str(metrics_csv_path))

    logger.info("MLflow run ID: %s", run.info.run_id)
    logger.info("MLflow experiment: backlink-pricing")

## 16. Save artifacts

Persist the trained model and training metadata to disk for downstream use in the prediction pipeline.

In [ ]:
# Save model.
model_path = MODELS_DIR / "xgb_backlink_pricing.joblib"
joblib.dump(best_xgb, model_path)
logger.info("Saved model to %s", model_path)

# Save training metadata.
training_metadata = {
    "model": "XGBRegressor",
    "random_seed": RANDOM_SEED,
    "n_optuna_trials": 100,
    "best_params": study.best_params,
    "feature_columns": FEATURE_COLS,
    "target": TARGET,
    "train_size": len(X_train),
    "test_size": len(X_test),
    "test_metrics": {
        k: round(v, 6) for k, v in all_metrics["XGBoost (tuned)"].items()
    },
    "all_model_metrics": {
        name: {k: round(v, 6) for k, v in m.items()}
        for name, m in all_metrics.items()
    },
}

metadata_path = MODELS_DIR / "training_metadata.json"
with open(metadata_path, "w") as f:
    json.dump(training_metadata, f, indent=2)
logger.info("Saved training metadata to %s", metadata_path)

# Save metrics summary CSV.
metrics_df.to_csv(MODELS_DIR / "metrics_summary.csv")
logger.info("Saved metrics summary to %s", MODELS_DIR / "metrics_summary.csv")

---

## 17. Summary

This notebook trained and evaluated five regression models on the engineered backlink pricing dataset, then analyzed the best model with SHAP for interpretability.

**Key findings:**
- **Baseline progression:** Mean baseline establishes the floor; Ridge captures linear signal; Random Forest and Gradient Boosting improve via non-linear splits.
- **XGBoost + Optuna:** Hyperparameter tuning over 100 trials yields the best RMSE and R-squared, confirming the value of systematic search.
- **Feature importance:** Domain Rating (DR) and log-traffic are the dominant predictors. TLD and country encodings provide secondary signal.
- **SHAP consistency:** SHAP values align with gain-based importance, and the beeswarm plot confirms that higher DR and traffic push prices upward.
- **Residuals:** Approximately symmetric and unbiased, with no systematic pattern against predicted values.

**Output artifacts:**
- `models/xgb_backlink_pricing.joblib` -- serialized best XGBoost model
- `models/training_metadata.json` -- hyperparameters, feature list, and test metrics
- `models/metrics_summary.csv` -- comparison table for all models
- `images/modeling/` -- all saved figures (comparison charts, scatter, residuals, importance, SHAP)
- `mlruns/` -- MLflow experiment tracking data